## For DNAase controls of 80K NGN2 derived neurons
- control groups covered: 'GC_DNase_positive', 'GC_DNase_negative_brain', 'GC_DNase_negative_blood', 'GC_DNase_positive_shuffeled', 'GC_DNase_negative_brain_shuffeled', 'GC_DNase_negative_blood_shuffeled'

In [3]:
from importlib import reload
import pandas as pd
import sys
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../../../00_helpful_functions/helpful_functions.py'>

In [8]:
# helpful functions

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


shuffled_group = ['GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled']

dnase_control_groups = ['GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood'] + shuffled_group


def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)
    Case: GC_DNase_positive_shuffeled: (all have ":chr" pattern)
            header: GC_DNase_positive_shuffeled:chr1:121484605-121484874_active_count_114
    """

    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled':
        return row
    name = row[col_name]

    if name.startswith('GC_DNase_positive:') or name.startswith('GC_DNase_negative_brain:') or name.startswith('GC_DNase_negative_blood:') or ('_shuffeled:' in name):
        row[col_ref] = 'GRCh37'
        row[col_category] = 'scrambled'
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
    return row


In [16]:
pre_metadata_df = hf.fasta_to_dataframe('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/renamed_design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions_REF_to_elements.fa', columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
pre_metadata_df

# # filter for underscore parsable headers
pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'].isin(dnase_control_groups)]
pre_metadata_df_filtered # 971

# add the columns of the metadata file
pre_metadata_df_filtered[col_category] = 'NA'
pre_metadata_df_filtered[col_class] = 'NA'
pre_metadata_df_filtered[col_source] = 'NA'
# all are GRCh37
pre_metadata_df_filtered[col_ref] = 'GRCh37'
pre_metadata_df_filtered[col_chr] = 'NA'
pre_metadata_df_filtered[col_start] = 'NA'
pre_metadata_df_filtered[col_end] = 'NA'
pre_metadata_df_filtered[col_strand] = 'NA'
pre_metadata_df_filtered[col_variant_class] = 'NA'
pre_metadata_df_filtered[col_variant_pos] = 'NA'
pre_metadata_df_filtered[col_SPDI] = 'NA'
pre_metadata_df_filtered[col_allele] = 'NA'
pre_metadata_df_filtered[col_info] = 'Coordinates are based on GRCh37 (wrong genome build)'

# parse regions from header
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_start_end_strand_control, axis=1)
pre_metadata_df_filtered

# remove adapter from sequence (15bp of start and end):
pre_metadata_df_filtered[col_sequence] = pre_metadata_df_filtered[col_sequence].apply(lambda x: x[15:-15])

/tmp/ipykernel_469357/1672550693.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pre_metadata_df_filtered[col_category] = 'NA'
/tmp/ipykernel_469357/1672550693.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pre_metadata_df_filtered[col_class] = 'NA'
/tmp/ipykernel_469357/1672550693.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.or

In [18]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

# Write DataFrame to TSV file
pre_metadata_df_filtered[interesting_columns].to_csv('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/dnase_controls/dnase_controls.metadata.tmp.tsv.gz', sep='\t', index=False, na_rep='NA', compression='gzip')
import os
os.system('zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/dnase_controls/dnase_controls.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/dnase_controls/dnase_controls.metadata.tsv.gz')

0